# Bài 4.1 · Dùng model pretrained trên ảnh thật

**Imagenette** là 10 lớp lấy từ ImageNet (cá tench, chó springer, máy cassette, cưa máy, nhà thờ, kèn French horn, xe rác, cây xăng, bóng golf, dù lượn), khoảng 13 000 ảnh chụp thật. Vì mười lớp này nằm sẵn trong 1000 lớp của ImageNet, một ResNet-50 đã học trên ImageNet dự đoán được ngay, **không cần train**.

Notebook làm bốn việc:

1. Nạp model và đọc “hợp đồng” của nó: kích thước ảnh, mean, std, thứ tự kênh, tên 1000 lớp.
2. Dự đoán top-5 cho vài ảnh và đo accuracy trên tập validation.
3. Cố ý làm sai hợp đồng từng chỗ một (quên chuẩn hóa, đảo kênh màu, để `train()`, ảnh quá nhỏ) và đo xem mỗi chỗ sai làm accuracy đổi bao nhiêu.
4. Đo thời gian và bộ nhớ khi tăng batch, so số thực 32 bit với 16 bit.

Bật **GPU** và **Internet** trong *Session options*. Chạy hết khoảng 5 phút.

In [ ]:
import time, random, tarfile, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader, Subset

random.seed(0); np.random.seed(0); torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| thiết bị:',
      torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')

## 1. Dữ liệu

Tải bản ảnh cạnh ngắn 320 pixel (khoảng 330 MB). Mỗi thư mục con mang mã WordNet của lớp; bảng dưới đổi mã ấy sang **chỉ số trong 1000 lớp ImageNet**, vì model trả về 1000 logits.

In [ ]:
URL = 'https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz'
root = Path('/tmp/imagenette')
if not (root / 'imagenette2-320').exists():
    root.mkdir(parents=True, exist_ok=True)
    tgz = root / 'data.tgz'
    urllib.request.urlretrieve(URL, tgz)
    with tarfile.open(tgz) as f:
        f.extractall(root)
data_dir = root / 'imagenette2-320'

WNID_TO_IMAGENET = {'n01440764': 0, 'n02102040': 217, 'n02979186': 482, 'n03000684': 491, 'n03028079': 497,
                    'n03394916': 566, 'n03417042': 569, 'n03425413': 571, 'n03445777': 574, 'n03888257': 701}
view_ds = torchvision.datasets.ImageFolder(data_dir / 'val')
imagenet_idx = torch.tensor([WNID_TO_IMAGENET[w] for w in view_ds.classes])
print(len(view_ds), 'ảnh validation,', len(view_ds.classes), 'lớp')

## 2. Model và hợp đồng của nó

`weights.transforms()` cho đúng phép tiền xử lý mà model đã thấy khi học: thu ảnh về 232, cắt giữa 224, đổi sang số thực trong [0, 1], rồi chuẩn hóa từng kênh bằng mean, std của ImageNet. Ảnh đọc bằng PIL theo thứ tự kênh RGB.

In [ ]:
weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V2
model = torchvision.models.resnet50(weights=weights).to(device).eval()
prep = weights.transforms()
categories = weights.meta['categories']
n_params = sum(p.numel() for p in model.parameters())
print(prep)
print(f'{n_params:,} tham số, khoảng {n_params * 4 / 2**20:.0f} MB ở float32')
print('10 lớp của Imagenette trong ImageNet:', [categories[i] for i in imagenet_idx])

## 3. Dự đoán top-5 cho vài ảnh

Softmax trên 1000 logits cho xác suất; `topk` lấy năm lớp lớn nhất. Model chưa từng được bảo rằng chỉ có 10 lớp, nên nó có thể đoán một lớp ngoài danh sách nhưng hợp lý (một giống chó khác, một loại xe khác).

In [ ]:
idx = random.sample(range(len(view_ds)), 6)
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, i in zip(axes.flat, idx):
    img, y = view_ds[i]
    with torch.inference_mode():
        p = model(prep(img)[None].to(device)).softmax(1)[0].cpu()
    top = p.topk(5)
    ax.imshow(img); ax.axis('off')
    ax.set_title('\n'.join(f'{categories[k]} {v:.2f}' for v, k in zip(top.values, top.indices)), fontsize=9)
plt.tight_layout(); plt.show()

## 4. Accuracy trên tập validation

Hai cách chấm. **1000 lớp**: đúng khi lớp lớn nhất trong 1000 lớp đúng là lớp của ảnh. **10 lớp**: chỉ so 10 logits của các lớp Imagenette, giống như biết trước đề chỉ có 10 đáp án.

In [ ]:
def evaluate(transform, train_mode=False, batch=128, n=None):
    ds = torchvision.datasets.ImageFolder(data_dir / 'val', transform=transform)
    if n is not None:
        ds = Subset(ds, np.random.default_rng(0).permutation(len(ds))[:n])
    dl = DataLoader(ds, batch, shuffle=False, num_workers=4, pin_memory=True)
    model.train(train_mode)
    hit1000 = hit10 = total = 0
    with torch.inference_mode():
        for x, y in dl:
            logits = model(x.to(device)).cpu()
            hit1000 += (logits.argmax(1) == imagenet_idx[y]).sum().item()
            hit10 += (logits[:, imagenet_idx].argmax(1) == y).sum().item()
            total += len(y)
    model.eval()
    return hit1000 / total, hit10 / total

t = time.time()
acc1000, acc10 = evaluate(prep)
print(f'đúng hợp đồng: accuracy 1000 lớp {acc1000:.3f} | 10 lớp {acc10:.3f} | {time.time() - t:.0f} giây')

## 5. Làm sai hợp đồng từng chỗ một

Mỗi dòng chỉ đổi **một** điều so với cách đúng. Model, trọng số và ảnh giữ nguyên.

- **Quên chuẩn hóa**: bỏ `Normalize`, đầu vào nằm trong [0, 1] thay vì quanh 0.
- **Đảo kênh**: đưa BGR (thứ tự của OpenCV) vào model học bằng RGB.
- **Để `train()`**: BatchNorm dùng thống kê của chính batch thay vì thống kê đã lưu; batch nhỏ 8 ảnh làm rõ chuyện này.
- **Ảnh quá nhỏ**: thu về 64 pixel rồi phóng lại 224, chi tiết mất hết.

Để nhanh, mỗi dòng chấm trên cùng 1000 ảnh validation.

In [ ]:
mean, std = prep.mean, prep.std
base = [T.Resize(232), T.CenterCrop(224), T.ToTensor()]
cases = [
    ('đúng hợp đồng', T.Compose(base + [T.Normalize(mean, std)]), False, 128),
    ('quên chuẩn hóa', T.Compose(base), False, 128),
    ('đảo kênh RGB → BGR', T.Compose(base + [T.Normalize(mean, std), T.Lambda(lambda x: x.flip(0))]), False, 128),
    ('để train(), batch 8', T.Compose(base + [T.Normalize(mean, std)]), True, 8),
    ('ảnh 64 pixel', T.Compose([T.Resize(64), T.CenterCrop(64), T.Resize(224)] + base[2:] + [T.Normalize(mean, std)]), False, 128),
]
rows = []
for name, tf, train_mode, b in cases:
    a1000, a10 = evaluate(tf, train_mode, b, n=1000)
    rows.append((name, a1000, a10))
    print(f'{name:22s} 1000 lớp {a1000:.3f} | 10 lớp {a10:.3f}')
pd.DataFrame(rows, columns=['cách', 'acc 1000 lớp', 'acc 10 lớp']).round(3)

## 6. Batch, thời gian và bộ nhớ

Trên GPU, batch lớn hơn tận dụng song song tốt hơn nên số ảnh mỗi giây tăng, đến khi bão hòa; bộ nhớ đỉnh tăng gần tỉ lệ với batch vì phải giữ activation của cả batch. `autocast` float16 thường nhanh hơn và tốn ít bộ nhớ hơn trên T4. Trên CPU, ô này chỉ đo thời gian.

In [ ]:
def bench(batch, half):
    x = torch.randn(batch, 3, 224, 224, device=device)
    if device == 'cuda':
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    with torch.inference_mode(), torch.autocast(device, dtype=torch.float16, enabled=half and device == 'cuda'):
        model(x)                                    # chạy nháp một lần
        if device == 'cuda': torch.cuda.synchronize()
        t = time.time()
        for _ in range(5):
            model(x)
        if device == 'cuda': torch.cuda.synchronize()
    sec = (time.time() - t) / 5
    mem = torch.cuda.max_memory_allocated() / 2**20 if device == 'cuda' else float('nan')
    return batch / sec, mem

rows = []
for half in ([False, True] if device == 'cuda' else [False]):
    for b in [1, 8, 32, 128]:
        ips, mem = bench(b, half)
        rows.append(('float16' if half else 'float32', b, round(ips), round(mem) if mem == mem else None))
pd.DataFrame(rows, columns=['kiểu số', 'batch', 'ảnh/giây', 'bộ nhớ đỉnh (MB)'])

## Đọc kết quả

Số dưới đây là của lần chạy mẫu trên GPU T4; chạy lại có thể lệch chút ít.

- **Không train gì** mà chọn đúng trong 10 lớp ở 99,8% ảnh validation. Trong cả 1000 lớp, model chỉ đúng 87,2%: khoảng 13% ảnh nó chọn một lớp ImageNet khác, dù vẫn xếp lớp đúng cao nhất trong 10 lớp của Imagenette. Xem top-5 ở mục 3 để thấy những lớp thay thế ấy là gì.
- **Làm sai hợp đồng** (trên 1000 ảnh, 1000 lớp): đúng hợp đồng 0,863; quên chuẩn hóa 0,850; đảo kênh màu 0,811; để `train()` với batch 8 còn 0,690; ảnh 64 pixel chỉ còn 0,377. Quên chuẩn hóa ở đây tụt ít, nhưng không có gì bảo đảm điều đó với model hay dữ liệu khác; hai lỗi `train()` và độ phân giải làm hỏng hẳn kết quả.
- **Batch và kiểu số**: với float32, số ảnh mỗi giây tăng từ 188 (batch 1) lên 387 (batch 128) và gần như bão hòa từ batch 32; bộ nhớ đỉnh tăng từ 135 lên 1455 MB. Với float16, batch 128 đạt 848 ảnh/giây, nhanh gấp khoảng 2,2 lần, và tốn 867 MB. Ở batch 1, float16 không nhanh hơn float32.

Gợi ý khi demo: đổi `n=1000` ở mục 5 thành cả tập, hoặc thử một model khác trong `torchvision.models` với đúng `weights.transforms()` của nó.